In [1]:
import xarray as xr
import matplotlib.pyplot as plt 
import seaborn as sns
import cartopy.crs as ccrs
import seaborn.objects as so
import numpy as np
from dask.distributed import LocalCluster, Client
import flox.xarray
import xesmf as xe
from affine import Affine

from tqdm.notebook import tqdm
import scipy
import dask.bag as db
from pyproj import Proj, Transformer, CRS
import geopandas as gpd
from rasterio import features
from shapely.geometry import shape

import pandas as pd
plt.style.use('robin')
from matplotlib.colors import LogNorm, Normalize
proj = ccrs.NorthPolarStereo(0,70)

def sigmoid(x):
    return 1/(1+np.exp(-x))

In [ ]:
# run_id = '3w874ud3'
# run_id = '0obl1joe'
run_id = '75kakckt'
# ds_attrs = xr.open_dataset(f'/Data/gfi/users/rogui7909/data/NN_outputs/attributions/newproj/{run_id}_attributions.nc', chunks = dict(time_of_event=20)).load()
ds_attrs = xr.open_dataset(f'/Data/gfi/users/rogui7909/data/NN_outputs/attributions/newproj/{run_id}_ALL_attributions_v2.nc').transpose('y','x',...)
ds_moran = xr.open_dataset(f'/Data/gfi/users/rogui7909/data/NN_outputs/attributions/newproj/{run_id}_moranI_lrp_normed_v2.nc') 
ds_attrs['moran_I'] = ds_moran.Moran_I

lrp_normed = ds_attrs.attributions_lrp 
lrp_normed = lrp_normed.sum('var_name')/lrp_normed.sum('var_name').clip(0).sum(['x','y'])
ds_attrs['lrp_normed'] = lrp_normed

# ig_normed = ds_attrs.attributions_ig 
# ig_normed = ig_normed/ig_normed.sum('var_name').max(['x','y'])
# # cycs = cycs.query("msl<100000")
gdf_cyclones = gpd.read_file(f'/Data/gfi/users/rogui7909/data/NN_outputs/attributions/newproj/{run_id}_cyclones_data.geojson')

In [ ]:
print(ds_attrs.moran_I.stack(sample=['time_of_event','timestep_past']).idxmax().values)
print(ds_attrs.moran_I.stack(sample=['time_of_event','timestep_past']).idxmin().values)
fig, axs = plt.subplot_mosaic("""ABCD""", figsize=(10,2.8), 
                              per_subplot_kw=dict(BCD=dict(projection=proj)))
ds_attrs.moran_I.stack(sample=['time_of_event','timestep_past']).idxmax()

ds_attrs.moran_I.plot.hist(ax=axs['A'], bins=30, color='.7')
ds_attrs.sel(time_of_event='2010-10-29', timestep_past=-4).lrp_normed.plot(ax=axs['B'], transform=proj, add_colorbar=False, cmap='icefire', vmax=.008)
ds_attrs.sel(time_of_event='2020-03-17', timestep_past=-1).lrp_normed.plot(ax=axs['C'], transform=proj, add_colorbar=False, cmap='icefire', vmax=.008)
ds_attrs.sel(time_of_event='2015-11-16', timestep_past=-2).lrp_normed.plot(ax=axs['D'], transform=proj, add_colorbar=False, cmap='icefire', vmax=.008)


lowest_moran = ds_attrs.sel(time_of_event='2010-10-29', timestep_past=-4).moran_I.values
median_moran = ds_attrs.sel(time_of_event='2020-03-17', timestep_past=-1).moran_I.values
highest_moran = ds_attrs.sel(time_of_event='2015-11-16', timestep_past=-2).moran_I.values

for letter in 'BCD':
    axs[letter].coastlines(color='.7', lw=.4, alpha=.5)
axs['A'].set_title('Histogram of values')
axs['A'].set(xlabel="Moran's I", ylabel='#', xlim=(.65,.9), ylim=(0,1000))
axs['B'].set_title(f'Lowest Moran\'s I: {lowest_moran:.02f}\n2010-10-29, D-4')
axs['C'].set_title(f'Median Moran\'s I: {median_moran:.02f}\n2020-03-17, D-1')
axs['D'].set_title(f'Highest Moran\'s I: {highest_moran:.02f}\n2015-11-16, D-2')
plt.tight_layout()
# plt.savefig('plots/moranI_examples_lrp_normed_v2.svg', dpi=300)